# GPU Validation: KV Cache Tier Persistence (TinyLlama-1.1B)

Reruns the end-to-end TinyLlama cache save/restore validation on a GPU, producing
the GPU-side numbers for the paper's break-even analysis.

**Before running:** `Runtime` → `Change runtime type` → select **T4 GPU** → Save.
Then `Runtime` → `Run all` (~10–15 min). The last cell downloads
`tinyllama_gpu_results.json` — that file is the deliverable.

Every step asserts its own success: if any cell shows an error, the run is
invalid — copy the error text rather than the downloaded file.

In [1]:
# Step 1: confirm a GPU is attached
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU: set Runtime -> Change runtime type -> T4 GPU, then rerun"
print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Mon Jul  6 15:30:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Step 2: clone the repo and install dependencies.
# transformers is PINNED to the version the integration script is validated
# against (Colab often ships a newer release with a changed cache API).
import os
if not os.path.exists('kv-cache-tier-persistence'):
    !git clone -q https://github.com/aditya-lawankar/kv-cache-tier-persistence.git
%cd kv-cache-tier-persistence
!pip install -q -e . 'transformers==4.51.*' accelerate

import transformers, zstandard, lz4
assert transformers.__version__.startswith('4.51'), f"transformers pin failed: {transformers.__version__} — rerun this cell (Colab sometimes needs a second pass after pip changes)"
print(f"transformers {transformers.__version__} | torch {torch.__version__}")

/content/kv-cache-tier-persistence
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for kv-cache-tier-persistence (pyproject.toml) ... done
transformers 4.51.3 | torch 2.11.0+cu128


In [3]:
# Step 3: run 1 — 256- and 512-token prompts, 3 trials each
!python benchmarks/tinyllama_integration.py --max-tokens 512 --trials 3 --output benchmarks/results/gpu_512
assert os.path.exists('benchmarks/results/gpu_512/tinyllama_integration_results.json'), \
    "Run 1 produced no results — scroll up in THIS cell for the traceback"


  TinyLlama KV Cache Persistence — Live Integration Test
  Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Device: Tesla T4 (CUDA, FP16)
tokenizer_config.json: 1.29kB [00:00, 5.59MB/s]
tokenizer.model: 100% 500k/500k [00:00<00:00, 1.14MB/s]
tokenizer.json: 1.84MB [00:00, 60.4MB/s]
special_tokens_map.json: 100% 551/551 [00:00<00:00, 4.50MB/s]
config.json: 100% 608/608 [00:00<00:00, 4.94MB/s]
2026-07-06 15:30:52.005554: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
model.safetensors: 100% 2.20G/2.20G [00:25<00:00, 88.0MB/s]
generation_config.json: 100% 124/124 [00:00<00:00, 839kB/s]
  Model config: 22 layers, 32 Q-heads, 4 KV-heads, head_dim=64
  Parameters: 1100.0M
  Tier storage: /tmp/kvcache_tinyllama_ppn40sez

  Building 512-token conver

In [4]:
# Step 4: run 2 — ~1800-token prompts (near TinyLlama's 2048 context limit)
!python benchmarks/tinyllama_integration.py --max-tokens 1800 --trials 3 --output benchmarks/results/gpu_1800
assert os.path.exists('benchmarks/results/gpu_1800/tinyllama_integration_results.json'), \
    "Run 2 produced no results — scroll up in THIS cell for the traceback"


  TinyLlama KV Cache Persistence — Live Integration Test
  Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Device: Tesla T4 (CUDA, FP16)
2026-07-06 15:31:55.425884: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
  Model config: 22 layers, 32 Q-heads, 4 KV-heads, head_dim=64
  Parameters: 1100.0M
  Tier storage: /tmp/kvcache_tinyllama_qy75wneg

  Building 1800-token conversation prompt...
Token indices sequence length is longer than the specified maximum sequence length for this model (2353 > 2048). Running this sequence through the model will result in indexing errors
  Actual prompt length: 2389 tokens

  ── Trial 1/3 (target: 256 tokens) ──
    Prompt: 598 tokens
    [1/5] Cold-start prefill... done (1590ms)
    [2/5] Extracting KV cache.

In [5]:
# Step 5: merge, summarize, and download the deliverable
import json, glob

rows = []
for path in sorted(glob.glob('benchmarks/results/gpu_*/tinyllama_integration_results.json')):
    rows += json.load(open(path))
assert rows, "No results found — a run cell above failed; do not use the downloaded file"
rows.sort(key=lambda r: r['prompt_tokens'])

with open('tinyllama_gpu_results.json', 'w') as f:
    json.dump(rows, f, indent=2)

print(f"{'tokens':>7} {'cold TTFT':>10} {'warm TTFT':>10} {'speedup':>8} {'save':>7} {'load':>7} {'match':>6} {'device':>7}")
for r in rows:
    print(f"{r['prompt_tokens']:>7} {r['cold_start_ttft_ms']:>8.0f}ms {r['warm_start_ttft_ms']:>8.0f}ms "
          f"{r['speedup_x']:>7.2f}x {r['save_latency_ms']:>5.0f}ms {r['load_latency_ms']:>5.0f}ms "
          f"{'  yes' if r['semantic_match'] else '   NO'} {r.get('device','?'):>7}")

from google.colab import files
files.download('tinyllama_gpu_results.json')

 tokens  cold TTFT  warm TTFT  speedup    save    load  match  device
    598     1590ms     1249ms    1.27x    53ms    21ms   yes    cuda
    598     1670ms     1438ms    1.16x    75ms    24ms   yes    cuda
    598     1267ms     1248ms    1.02x    50ms    21ms   yes    cuda
    598     2114ms     1247ms    1.69x    53ms    21ms   yes    cuda
    598     1544ms     1592ms    0.97x    73ms    17ms   yes    cuda
    598     1270ms     1233ms    1.03x    46ms    17ms   yes    cuda
    598     1253ms     1216ms    1.03x    49ms    17ms   yes    cuda
    598     1264ms     1211ms    1.04x    46ms    16ms   yes    cuda
    598     1267ms     1594ms    0.80x    45ms    18ms   yes    cuda
   2389     1508ms     1204ms    1.25x   238ms    97ms    NO    cuda
   2389     1538ms     1426ms    1.08x   236ms    98ms    NO    cuda
   2389     1917ms     1209ms    1.59x   236ms   100ms    NO    cuda


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>